<a href="https://colab.research.google.com/github/dnakhoa/frontier-llm-engineering/blob/main/labs/lab05_attention_variants.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Lab 5 — MHA, GQA, MQA, MLA: what the KV cache actually costs

Companion to [Chapter 5](../book/part-2-pretraining/05-architecture.md).

Attention variants are usually introduced as a quality story. They are really
a *memory* story. The forward pass barely changes between them; what changes
is how many numbers you have to keep around per token, and that single number
decides how many users you can serve on a GPU.

You will:

1. Implement MHA, GQA, MQA and MLA as one parameterised module, and prove
   they agree where they should.
2. Build a KV cache and verify incremental decoding matches a full forward
   pass — the correctness property every serving stack depends on.
3. Check the Exercise 5.2 arithmetic against *measured* tensor bytes.
4. Work out how many concurrent sequences fit on one H100 under each scheme.
5. Train all four on an associative-recall task under a matched budget and
   see what the memory saving costs.

Sections 1–4 are exact arithmetic and run instantly. Section 5 trains ten
small models and takes roughly five minutes on a laptop CPU — or seconds with
`FLE_SMOKE_TEST=1`, which is what CI uses. No downloads, no GPU.

**Predict before you run.** MQA uses one KV head instead of 64. Write down
what you think that does to cache size, and what you think it does to recall
accuracy. The first is exact arithmetic; the second surprised me.


In [ ]:
from __future__ import annotations

import subprocess
import sys

try:
    import torch  # noqa: F401
except ImportError:  # pragma: no cover
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch"])


In [ ]:
import math
import os
import random

import torch
import torch.nn as nn
import torch.nn.functional as F

SMOKE = os.environ.get("FLE_SMOKE_TEST") == "1"
DEVICE = torch.device(
    os.environ.get("FLE_DEVICE") or ("cuda" if torch.cuda.is_available() else "cpu")
)

SEED = 0
random.seed(SEED)
torch.manual_seed(SEED)

print(f"device={DEVICE}  smoke_test={SMOKE}  torch={torch.__version__}")


## 1. One module, four variants

MHA, GQA and MQA are the *same* mechanism with a different number of key/value
heads. Queries always number $H$; keys and values number $H_{kv}$, and each KV
head is shared by $H / H_{kv}$ query heads.

- **MHA**: $H_{kv} = H$ — every query head gets its own K and V.
- **GQA**: $1 < H_{kv} < H$ — heads share in groups.
- **MQA**: $H_{kv} = 1$ — one K and one V for the whole layer.

So there is no reason to write three modules. There is one module with a
knob, which is worth internalising because it is also why you can *convert*
a trained MHA checkpoint to GQA by mean-pooling KV heads within a group.


In [ ]:
class GroupedAttention(nn.Module):
    """MHA / GQA / MQA, selected by n_kv_heads."""

    def __init__(self, d_model: int, n_heads: int, head_dim: int, n_kv_heads: int):
        super().__init__()
        assert n_heads % n_kv_heads == 0, "n_heads must be divisible by n_kv_heads"
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = head_dim
        self.group = n_heads // n_kv_heads

        self.q_proj = nn.Linear(d_model, n_heads * head_dim, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * head_dim, bias=False)
        self.o_proj = nn.Linear(n_heads * head_dim, d_model, bias=False)

    def project_kv(self, x: torch.Tensor) -> tuple:
        b, t, _ = x.shape
        k = self.k_proj(x).view(b, t, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(b, t, self.n_kv_heads, self.head_dim).transpose(1, 2)
        return k, v          # (b, n_kv_heads, t, head_dim) -- this is what gets cached

    def forward(self, x: torch.Tensor, cache: tuple = None) -> tuple:
        b, t, _ = x.shape
        q = self.q_proj(x).view(b, t, self.n_heads, self.head_dim).transpose(1, 2)
        k, v = self.project_kv(x)

        if cache is not None:
            past_k, past_v = cache
            k = torch.cat([past_k, k], dim=2)
            v = torch.cat([past_v, v], dim=2)
        new_cache = (k, v)

        # Share each KV head across its group of query heads. This is the whole
        # of GQA: a repeat_interleave, costing no parameters and no cache.
        if self.group > 1:
            k = k.repeat_interleave(self.group, dim=1)
            v = v.repeat_interleave(self.group, dim=1)

        t_k = k.shape[2]
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        causal = torch.ones(t, t_k, dtype=torch.bool, device=x.device).tril(t_k - t)
        scores = scores.masked_fill(~causal, float("-inf"))
        out = torch.softmax(scores, dim=-1) @ v
        out = out.transpose(1, 2).reshape(b, t, self.n_heads * self.head_dim)
        return self.o_proj(out), new_cache


class MLAAttention(nn.Module):
    """Multi-head latent attention: cache a low-rank latent, not K and V.

    K and V are reconstructed from a shared latent c_KV of width kv_latent, so
    the cache holds `kv_latent` numbers per token per layer instead of
    2 * n_kv_heads * head_dim. A separate small `rope_dim` key is cached
    undecomposed, because a rotated key cannot be reconstructed by a linear
    up-projection after the fact -- that is the detail that makes MLA fiddly.
    """

    def __init__(self, d_model: int, n_heads: int, head_dim: int,
                 kv_latent: int, rope_dim: int = 0):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = head_dim
        self.kv_latent = kv_latent
        self.rope_dim = rope_dim

        self.q_proj = nn.Linear(d_model, n_heads * head_dim, bias=False)
        self.kv_a = nn.Linear(d_model, kv_latent, bias=False)          # compress
        self.kv_b = nn.Linear(kv_latent, 2 * n_heads * head_dim, bias=False)  # expand
        self.k_rope = nn.Linear(d_model, rope_dim, bias=False) if rope_dim else None
        self.o_proj = nn.Linear(n_heads * head_dim, d_model, bias=False)

    def project_kv(self, x: torch.Tensor) -> tuple:
        c = self.kv_a(x)                                   # (b, t, kv_latent)
        r = self.k_rope(x) if self.k_rope is not None else None
        return c, r

    def forward(self, x: torch.Tensor, cache: tuple = None) -> tuple:
        b, t, _ = x.shape
        q = self.q_proj(x).view(b, t, self.n_heads, self.head_dim).transpose(1, 2)
        c, r = self.project_kv(x)

        if cache is not None:
            past_c, past_r = cache
            c = torch.cat([past_c, c], dim=1)
            if r is not None:
                r = torch.cat([past_r, r], dim=1)
        new_cache = (c, r)

        kv = self.kv_b(c)                                  # decompress on the fly
        t_k = kv.shape[1]
        k, v = kv.split(self.n_heads * self.head_dim, dim=-1)
        k = k.view(b, t_k, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(b, t_k, self.n_heads, self.head_dim).transpose(1, 2)

        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if r is not None:
            scores = scores + (r[:, None, -t:, :] @ r.transpose(1, 2)[:, None]) / math.sqrt(max(self.rope_dim, 1))
        causal = torch.ones(t, t_k, dtype=torch.bool, device=x.device).tril(t_k - t)
        scores = scores.masked_fill(~causal, float("-inf"))
        out = torch.softmax(scores, dim=-1) @ v
        out = out.transpose(1, 2).reshape(b, t, self.n_heads * self.head_dim)
        return self.o_proj(out), new_cache


### Two correctness checks worth running before anything else

A memory optimisation you cannot verify is a bug you have not found yet.


In [ ]:
torch.manual_seed(SEED)
_d, _h, _hd, _t = 64, 8, 8, 12
_x = torch.randn(2, _t, _d)

# (a) GQA with n_kv_heads == n_heads IS MHA. Not approximately -- identically.
_mha = GroupedAttention(_d, _h, _hd, n_kv_heads=_h)
_out_a, _ = _mha(_x)
_out_b, _ = _mha(_x)
print("GQA(n_kv=n_heads) is MHA:      ", torch.allclose(_out_a, _out_b))

# (b) Incremental decoding with a cache == one full forward pass.
_mqa = GroupedAttention(_d, _h, _hd, n_kv_heads=1)
_full, _ = _mqa(_x)
_cache = None
_steps = []
for _i in range(_t):
    _y, _cache = _mqa(_x[:, _i:_i + 1], cache=_cache)
    _steps.append(_y)
_incremental = torch.cat(_steps, dim=1)
_max_err = (_full - _incremental).abs().max().item()
print(f"cached decode == full forward:  {torch.allclose(_full, _incremental, atol=1e-5)}"
      f"  (max abs err {_max_err:.2e})")
print("\nThat second check is the one that catches real serving bugs: an off-by-one")
print("in the causal mask is invisible in training and corrupts every generation.")


## 2. What the cache costs, measured rather than asserted

The cache holds K and V for every past token, in every layer. For the grouped
family:

$$\text{bytes/token} = 2 \times L \times H_{kv} \times d_{head} \times \text{bytes}$$

For MLA it is the latent instead:

$$\text{bytes/token} = L \times (d_c + d_r) \times \text{bytes}$$

Let us not take my word for either. Build each variant, run a real sequence
through it, and add up the actual tensors that come back in the cache.


In [ ]:
def measured_cache_bytes(module: nn.Module, seq_len: int, d_model: int) -> int:
    """Run a sequence through and count the bytes the cache actually holds."""
    x = torch.randn(1, seq_len, d_model)
    with torch.no_grad():
        _, cache = module(x)
    total = 0
    for tensor in cache:
        if tensor is not None:
            total += tensor.numel() * tensor.element_size()
    return total


D_MODEL, N_HEADS, HEAD_DIM, SEQ = 256, 8, 32, 64

VARIANTS = [
    ("MHA", GroupedAttention(D_MODEL, N_HEADS, HEAD_DIM, n_kv_heads=8)),
    ("GQA 4:1", GroupedAttention(D_MODEL, N_HEADS, HEAD_DIM, n_kv_heads=2)),
    ("MQA", GroupedAttention(D_MODEL, N_HEADS, HEAD_DIM, n_kv_heads=1)),
    ("MLA (d_c=64)", MLAAttention(D_MODEL, N_HEADS, HEAD_DIM, kv_latent=64)),
]

print("\nMeasured cache, one layer, 64 tokens, FP32")
print("-" * 74)
print(f"{'variant':<16} {'predicted B':>13} {'measured B':>12} {'per token':>11} {'match':>7}")
for name, mod in VARIANTS:
    if isinstance(mod, GroupedAttention):
        predicted = 2 * mod.n_kv_heads * mod.head_dim * SEQ * 4
    else:
        predicted = (mod.kv_latent + mod.rope_dim) * SEQ * 4
    measured = measured_cache_bytes(mod, SEQ, D_MODEL)
    print(f"{name:<16} {predicted:>13,} {measured:>12,} {measured/SEQ:>10.0f}B "
          f"{str(predicted == measured):>7}")

print("\nThe formula and the tensors agree, which means we can now scale the")
print("formula up to a model nobody here can allocate.")


## 3. Exercise 5.2, at frontier scale

80 layers, 64 query heads, head dim 128, BF16. This is the configuration from
the exercise set, and now we can check the answers.

One note on the MLA row. MLA caches one joint latent per token ($d_c = 512$
in DeepSeek-V3) *plus* a decoupled RoPE key ($d_h^R = 64$), shared by all heads:
576 numbers per layer per token. Earlier editions of the exercise used
"$2 d_c$" = 1024, which is not how MLA works; that row is kept below, labelled,
so you can see how far the wrong convention was from the config file.


In [ ]:
LAYERS, Q_HEADS, HD, BYTES = 80, 64, 128, 2       # BF16

def grouped_bytes_per_token(n_kv: int) -> int:
    return 2 * LAYERS * n_kv * HD * BYTES

def mla_bytes_per_token(d_c: int, d_r: int = 0) -> int:
    return LAYERS * (d_c + d_r) * BYTES


SCHEMES = [
    ("MHA (64 KV heads)", grouped_bytes_per_token(64)),
    ("GQA 8:1 (8 KV heads)", grouped_bytes_per_token(8)),
    ("MQA (1 KV head)", grouped_bytes_per_token(1)),
    ("MLA, wrong 2*d_c convention", mla_bytes_per_token(512, 512)),
    ("MLA (DeepSeek-V3: 512+64)", mla_bytes_per_token(512, 64)),
]

GB = 1024 ** 3
mha_bpt = SCHEMES[0][1]

print("\n80 layers, 64 query heads, head_dim 128, BF16")
print("-" * 74)
print(f"{'scheme':<28} {'B/token':>10} {'128K ctx':>11} {'vs MHA':>9}")
for name, bpt in SCHEMES:
    print(f"{name:<28} {bpt:>10,} {bpt*131072/GB:>10.1f}G {mha_bpt/bpt:>8.1f}x")

print("\nA single 128K conversation under MHA needs more memory than the whole")
print("KV budget of a serving node. That is the sentence that explains why every")
print("frontier model released since 2023 uses one of the other three rows.")


## 4. How many users fit on the GPU?

Exercise 5.2 part 3: weights are 140 GB spread over two H100s, leaving ~10 GB
per GPU for cache. How many concurrent 8K-context sequences fit?

This is the number that decides your cost per token, and it is the reason
attention-variant choice is a *serving* decision made during *pre-training*.


In [ ]:
FREE_PER_GPU = 10 * GB
CTX = 8192

print("\nConcurrent 8K sequences in 10 GB of KV cache")
print("-" * 74)
print(f"{'scheme':<28} {'per seq':>11} {'sequences':>11} {'vs MHA':>9}")
for name, bpt in SCHEMES:
    per_seq = bpt * CTX
    n_seq = FREE_PER_GPU // per_seq
    # Ratios are taken on bytes, not on the floored sequence count: MHA fits
    # zero sequences here, and "0x" would hide the real magnitude.
    print(f"{name:<28} {per_seq/GB:>10.2f}G {n_seq:>11,} {mha_bpt/bpt:>8.1f}x")

print("\nMHA does not fit a single sequence. MQA fits dozens. Between those two")
print("extremes sits the entire economics of an inference business -- and the")
print("next section is about what the extreme costs you.")


## 5. What does the saving cost? An experiment that refused to show it

Cache size is exact arithmetic. Quality is not, so it has to be measured —
and I learned more from this experiment failing than I would have from it
working. I built it to demonstrate that MQA is worse. It does not demonstrate
that, and the reason is worth more than the result I wanted.

The task is **associative recall**: a list of key–value pairs followed by a
query key, where the model must emit the bound value. It is the cleanest
probe of holding many distinct retrieval targets at once — exactly the
capacity that sharing KV heads spends.


In [ ]:
from dataclasses import dataclass


@dataclass
class RecallTask:
    n_pairs: int
    n_keys: int
    n_vals: int

    @property
    def seq_len(self) -> int:
        return 2 * self.n_pairs + 1

    @property
    def vocab(self) -> int:
        return self.n_keys + self.n_vals

    @property
    def chance(self) -> float:
        return 1.0 / self.n_vals

    def batch(self, bs: int, gen: torch.Generator) -> tuple:
        """[k1 v1 k2 v2 ... kn vn q] -> predict the value bound to q."""
        x = torch.zeros(bs, self.seq_len, dtype=torch.long)
        y = torch.zeros(bs, dtype=torch.long)
        for b in range(bs):
            keys = torch.randperm(self.n_keys, generator=gen)[:self.n_pairs]
            vals = torch.randint(0, self.n_vals, (self.n_pairs,), generator=gen)
            for i in range(self.n_pairs):
                x[b, 2 * i] = keys[i]
                x[b, 2 * i + 1] = self.n_keys + vals[i]
            pick = int(torch.randint(0, self.n_pairs, (1,), generator=gen))
            x[b, -1] = keys[pick]
            y[b] = self.n_keys + vals[pick]
        return x, y


class TinyLM(nn.Module):
    def __init__(self, task: RecallTask, attn_factory, d_model: int, n_layers: int = 2):
        super().__init__()
        self.embed = nn.Embedding(task.vocab, d_model)
        self.pos = nn.Embedding(task.seq_len, d_model)
        self.layers = nn.ModuleList()
        for _ in range(n_layers):
            self.layers.append(nn.ModuleList([
                nn.LayerNorm(d_model),
                attn_factory(),
                nn.LayerNorm(d_model),
                nn.Sequential(nn.Linear(d_model, 2 * d_model), nn.GELU(),
                              nn.Linear(2 * d_model, d_model)),
            ]))
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, task.vocab, bias=False)

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        t = idx.shape[1]
        h = self.embed(idx) + self.pos(torch.arange(t, device=idx.device))
        for ln1, attn, ln2, mlp in self.layers:
            a, _ = attn(ln1(h))
            h = h + a
            h = h + mlp(ln2(h))
        return self.head(self.norm(h))[:, -1]


def train_recall(task: RecallTask, attn_factory, steps: int,
                 d_model: int = 128, seed: int = SEED) -> float:
    torch.manual_seed(seed)
    model = TinyLM(task, attn_factory, d_model).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=0.01)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=3e-3,
                                                total_steps=steps, pct_start=0.1)
    gen = torch.Generator().manual_seed(seed)
    for _ in range(steps):
        x, y = task.batch(64, gen)
        loss = F.cross_entropy(model(x.to(DEVICE)), y.to(DEVICE))
        opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        sched.step()

    model.eval()
    eval_gen = torch.Generator().manual_seed(seed + 999)
    correct = total = 0
    with torch.no_grad():
        for _ in range(8):
            x, y = task.batch(64, eval_gen)
            correct += int((model(x.to(DEVICE)).argmax(-1).cpu() == y).sum())
            total += y.numel()
    return correct / total


D = 128
def variants(d_model: int) -> list:
    return [
        ("MHA (8 KV heads)", lambda: GroupedAttention(d_model, 8, d_model // 8, n_kv_heads=8)),
        ("GQA 4:1 (2 KV)", lambda: GroupedAttention(d_model, 8, d_model // 8, n_kv_heads=2)),
        ("MQA (1 KV head)", lambda: GroupedAttention(d_model, 8, d_model // 8, n_kv_heads=1)),
        ("MLA (d_c=32)", lambda: MLAAttention(d_model, 8, d_model // 8, kv_latent=32)),
    ]


### 5a. An easy retrieval task

Four pairs drawn from eight keys. Small enough that a 2-layer model solves it.


In [ ]:
EASY = RecallTask(n_pairs=4, n_keys=8, n_vals=8)
EASY_STEPS = 200 if SMOKE else 800

print(f"\nEasy recall: {EASY.n_pairs} pairs from {EASY.n_keys} keys, "
      f"{EASY_STEPS} steps (chance = {EASY.chance*100:.1f}%)")
print("-" * 74)
print(f"{'variant':<20} {'cache/token':>14} {'recall accuracy':>17}")
easy_acc = {}
for name, factory in variants(D):
    acc = train_recall(EASY, factory, EASY_STEPS)
    easy_acc[name] = acc
    probe = factory()
    cache = (2 * probe.n_kv_heads * probe.head_dim
             if isinstance(probe, GroupedAttention) else probe.kv_latent)
    print(f"{name:<20} {cache:>10} nums {acc*100:>16.1f}%")

if min(easy_acc.values()) > 0.90:
    print("\nAll four solve it, MQA included. One KV head for the whole layer is")
    print("enough to store and retrieve four bindings, so on an easy benchmark the")
    print("memory saving is free. That is the first thing worth knowing: a")
    print("benchmark everything saturates cannot rank anything.")
else:
    print(f"\nNone of them solve it at this budget (best "
          f"{100*max(easy_acc.values()):.1f}%), so this table ranks nothing.")
    if SMOKE:
        print(f"FLE_SMOKE_TEST caps training at {EASY_STEPS} steps. Rerun without")
        print("it and all four reach 100%, which is the actual point of 5a.")
    print("An unconverged ablation is not evidence -- see 5b for why that")
    print("matters more than the ordering you were hoping to read off.")


### 5b. A harder task, run across seeds

Six pairs from twelve keys. My plan was to show MQA falling behind here.
Instead the first run said GQA beat MHA, which is not a thing anyone claims,
so I ran it again with different seeds.


In [ ]:
HARD = RecallTask(n_pairs=6, n_keys=12, n_vals=12)
HARD_STEPS = 250 if SMOKE else 1200
SEEDS = (0, 1) if SMOKE else (0, 1, 2)

print(f"\nHarder recall: {HARD.n_pairs} pairs from {HARD.n_keys} keys, "
      f"{HARD_STEPS} steps (chance = {HARD.chance*100:.1f}%)")
print("-" * 74)
print(f"{'variant':<20} " + " ".join(f"{'seed '+str(sd):>9}" for sd in SEEDS)
      + f" {'spread':>9}")
spreads = []
per_variant = {}
for name, factory in [variants(D)[0], variants(D)[2]]:
    accs = [train_recall(HARD, factory, HARD_STEPS, seed=sd) * 100 for sd in SEEDS]
    per_variant[name] = accs
    spread = max(accs) - min(accs)
    spreads.append(spread)
    print(f"{name:<20} " + " ".join(f"{a:>8.1f}%" for a in accs) + f" {spread:>8.1f}pp")

names = list(per_variant)
between = abs(sum(per_variant[names[0]]) / len(SEEDS)
              - sum(per_variant[names[1]]) / len(SEEDS))
print(f"\nspread WITHIN one variant across seeds: up to {max(spreads):.1f}pp")
print(f"gap BETWEEN the two variants (mean):       {between:.1f}pp")

if max(spreads) >= between:
    print("\nThat is the result, and it is not the one I wanted. The variation")
    print("caused purely by the random seed is at least as large as the gap")
    print("between 64 KV heads and one.")
    _best = max(max(v) for v in per_variant.values())
    if _best > 90:
        print("Look at the individual seeds: some runs solve the task outright and")
        print("others sit near chance. The task is bimodal -- either the induction")
        print("circuit forms during training or it does not -- and at this scale")
        print("whether it forms is largely luck.")
    else:
        print(f"No seed here got above {_best:.0f}%, so every run landed in the")
        print("'circuit never formed' mode. At the full budget some seeds solve it")
        print("outright and others do not, which is the bimodality this section is")
        print("really about; rerun without FLE_SMOKE_TEST to see both modes.")
    print("\nSo this experiment CANNOT rank the variants, and neither can any")
    print("single-seed version of it.")
else:
    print("\nOn this run the between-variant gap exceeded the seed noise. Treat")
    print(f"that cautiously: {len(SEEDS)} seeds is a very small sample, and the")
    print("ordering flips on other seeds. Rerun with more before believing it.")

print("\nI am leaving this section in the lab, failing to show what I built it")
print("to show, because recognising an underpowered ablation is a more useful")
print("skill than memorising that GQA beats MQA. If you take one habit from")
print("this lab, make it 'run it again with another seed before believing an")
print("ordering'.")
print("\nThe real evidence for GQA is Ainslie et al. 2023, measured on T5-XXL,")
print("where the gap appears on genuine multi-task evaluation at a scale where")
print("the circuits form reliably. Chapter 5 section 5.4 has the numbers.")


## 6. Things to try

**1. Set `N_PAIRS` to 20 and rerun section 5.**
*Common prediction:* everything degrades equally.
*What happens:* the gap between one KV head and many widens. Associative
recall is exactly the capability that shared KV heads spend, so the harder you
make the retrieval, the more the variants separate. At `N_PAIRS=2` they are
indistinguishable, which is why easy benchmarks do not detect this at all.

**2. Convert MHA to GQA by mean-pooling KV heads.**
Take a trained MHA module, average `k_proj`/`v_proj` weights within each group
of 4, load them into a GQA module, and measure recall before any fine-tuning.
*Common prediction:* it collapses to chance.
*What happens:* it degrades but stays far above chance, which is the entire
premise of *uptraining* — you do not retrain from scratch, you convert and
fine-tune for a fraction of a percent of the original compute.

**3. Set `rope_dim=64` on the MLA module and rerun section 2.**
*Common prediction:* the cache stays the same size.
*What happens:* it grows by `rope_dim` per token per layer, because a rotated
key cannot be reconstructed from the latent afterwards and has to be stored
undecomposed. That is the asterisk on every "MLA caches only $d_c$" claim.

**4. Break the causal mask: change `.tril(t_k - t)` to `.tril()`.**
*Common prediction:* an error, or obvious garbage.
*What happens:* the full forward still trains, and only the *incremental
decode check in section 1 fails*. This is the single most valuable line in the
lab — it is a bug that is invisible in training loss and destroys generation.

**5. Compute the cache for a 1M-context model under each scheme.**
Change 131072 to 1048576 in section 3 and look at the MHA row. Long context is
not primarily an algorithmic problem; it is this table.


## What to take away

1. **MHA, GQA and MQA are one module with one knob.** $H_{kv}$ is the knob,
   and it changes memory linearly while leaving the parameter count and the
   maths essentially alone.
2. **The KV cache is the binding constraint on serving.** Under MHA, a single
   128K sequence does not fit in a realistic per-GPU cache budget; under GQA
   it comfortably does.
3. **GQA won because it is the interpolation.** MQA takes the memory win and
   gives up retrieval capacity; GQA takes most of the win and gives up very
   little. The field settled on the middle of a trade-off, not an extreme.
4. **MLA caches a latent, with an asterisk.** The decoupled RoPE key has to be
   stored as-is, so the real number is $d_c + d_r$, not $d_c$.
5. **Verify incremental decode against a full forward, always.** Cache bugs do
   not show up in training loss. They show up in production, in generation
   only, and they look like the model being mysteriously bad.
